In [0]:
%run ./P5_00_Config

In [0]:
%pip install xgboost

from xgboost import XGBRegressor
import mlflow
import pandas as pd
import numpy as np

In [0]:
df_silver = spark.table(SILVER_TABLE)

df_automotive = df_silver.filter(df_silver.family == 'AUTOMOTIVE')

df_automotive = df_automotive.toPandas()

df_automotive["date"] = pd.to_datetime(df_automotive["date"])

In [0]:
df_automotive = df_automotive.sort_values("date")

In [0]:
#Lag features
df_automotive["lag_7"] = df_automotive["sales"].shift(7)
df_automotive["lag_14"] = df_automotive["sales"].shift(14)
df_automotive["lag_28"] = df_automotive["sales"].shift(28)

#Rolling mean features
df_automotive["rolling_mean_7"] = df_automotive["sales"].rolling(7).mean()
df_automotive["rolling_mean_28"] = df_automotive["sales"].rolling(28).mean()

#Drop rows with nulls generated by lags
df_automotive = df_automotive.dropna()

In [0]:
FEATURES = ["year", "month", "dayofweek", "onpromotion", "is_holiday", "dcoilwtico", "lag_7", "lag_14", "lag_28", "rolling_mean_7", "rolling_mean_28"]

X = df_automotive[FEATURES]
y = df_automotive["sales"]

In [0]:
with mlflow.start_run(run_name="xgboost_automotive"):

    #Train the model
    model = XGBRegressor(n_estimators=100, learning_rate=0.1, random_state=42)
    model.fit(X, y)

    #Generate predictions on training data
    y_pred = model.predict(X)

    #Calculate MAE
    mae = np.abs(y - y_pred).mean()

    #Log parameters to MLflow
    mlflow.log_param("n_estimators", 100)
    mlflow.log_param("learning_rate", 0.1)
    mlflow.log_param("family", "AUTOMOTIVE")
    mlflow.log_param("features", FEATURES)

    #Log metric to MLflow
    mlflow.log_metric("mae", mae)
    print(f"MAE: {mae:.2f}")

In [0]:
df_gold = pd.DataFrame({
    "ds": df_automotive["date"],
    "yhat": y_pred,
    "yhat_upper": None,
    "yhat_lower": None,
    "family": "AUTOMOTIVE",
    "model": "xgboost"
})

df_gold = spark.createDataFrame(df_gold)
df_gold.write.format("delta").mode("append").saveAsTable(GOLD_TABLE)